# Estymator smażenia

Model termiczny smażenia kotleta na patelni. Na wejściu: rodzaj mięsa, grubość i średnica
kotleta, materiał i wymiary patelni, rodzaj i moc źródła ciepła (gaz / płyta elektryczna /
indukcja) oraz warunki otoczenia. Na wyjściu: pole temperatury w kotlecie w czasie, temperatura
patelni, ubytek wody, wskaźnik zbrązowienia powierzchni, skumulowana letalność mikrobiologiczna
oraz rekomendowany czas smażenia i moment obracania.

Dane materiałowe pochodzą z zewnętrznych źródeł (tabele ASHRAE / Choi–Okos, USDA FSIS,
pomiary sprawności płyt kuchennych) i są trzymane w plikach CSV obok notebooka, nie w kodzie.

| Plik | Zawartość |
|---|---|
| `heat_capacities/food.csv` | skład podstawowy mięs (woda / białko / tłuszcz / popiół) |
| `heat_capacities/component_properties_choi_okos.csv` | współczynniki modelu Choi–Okos (SI) |
| `heat_capacities/pan_capacities.csv` | właściwości cieplne materiałów patelni |
| `data/heat_sources.csv` | moc i zmierzona sprawność palników i płyt |
| `data/usda_appendix_a_lethality.csv` | tabele czas–temperatura USDA FSIS Appendix A |

## 1. Model fizyczny

### 1.1. Twoje założenie i jego korekta

Zakładałeś, że ciepło rozchodzi się głównie od patelni pionowo w górę. **To jest poprawne
założenie co do kierunku transportu**, ale wymaga trzech uściśleń.

**(a) Dlaczego boki naprawdę można pominąć.** Stosunek pola bocznego do pola styku z patelnią
wynosi $2\pi R L / (\pi R^2) = 2L/R$, czyli dla $L = 15$ mm i $R = 45$ mm aż $0{,}67$ — sam
stosunek pól niczego nie przesądza. Rozstrzyga stosunek *strumieni*: na dole współczynnik
przejmowania ciepła wynosi $h_c \approx 900\ \mathrm{W/(m^2 K)}$, na boku (konwekcja swobodna
w powietrzu) $h \approx 12\ \mathrm{W/(m^2 K)}$. Moc wchodząca bokiem stanowi więc około
$0{,}67 \times 12/900 \approx 0{,}9\%$ mocy wchodzącej dnem. Dopiero to uzasadnia model 1D.
Notebook liczy tę liczbę dla zadanych parametrów w sekcji 5.

**(b) To nie jest „rozchodzenie się ciepła”, tylko przewodzenie z oporem kontaktowym.**
Mięso nie przylega do patelni idealnie: kontakt jest punktowy, a między powierzchniami jest
warstwa tłuszczu i pary. Opisuje to kontaktowy współczynnik przejmowania ciepła
$h_c = 200\text{–}1200\ \mathrm{W/(m^2 K)}$ (Dagerskog 1985, pomiary dla płyt kontaktowych).
Liczba Biota mięsa $\mathrm{Bi} = h_c L / k \approx 900 \cdot 0{,}015 / 0{,}5 \approx 27 \gg 1$
oznacza, że **nie wolno** traktować kotleta jako ciała o jednolitej temperaturze — trzeba
rozwiązać równanie przewodnictwa.

**(c) Największym odbiornikiem energii nie jest ogrzanie mięsa, tylko odparowanie wody.**
Podgrzanie kotleta 100 g od 6 °C do 70 °C to około $0{,}1 \cdot 3500 \cdot 64 \approx 22$ kJ.
Odparowanie 20 % masy jako wody to $0{,}02 \cdot 2{,}26\cdot 10^6 \approx 45$ kJ — dwa razy
więcej. Model bez parowania zawyża tempo nagrzewania o rząd wielkości na powierzchni i nie
odtwarza plateau przy 100 °C, bez którego nie ma ani skórki, ani reakcji Maillarda.

**(d) Patelnia nie jest izotermiczna.** Pod kotletem powstaje zimne miejsce. Tempo jego
regeneracji zależy od rozpływu bocznego ($k\delta$) i od pojemności cieplnej ($\rho c_p \delta$)
dna. To jest właściwy powód, dla którego materiał patelni w ogóle ma znaczenie — i dlatego
patelnia jest w modelu *dwuwęzłowa*, mimo że mięso jest 1D.

### 1.2. Równania

**Mięso** — jednowymiarowe przewodzenie wzdłuż osi $z$ (od dna, $z=0$, do góry, $z=L$), ze
współczynnikami zależnymi od temperatury:

$$\rho(T)\, c_p(T)\, \frac{\partial T}{\partial t}
 = \frac{\partial}{\partial z}\!\left( k(T)\, \frac{\partial T}{\partial z} \right),
 \qquad 0 < z < L .$$

**Warunek brzegowy na dole** (styk z patelnią) — przewodzenie kontaktowe minus ciepło
utajone odparowania na froncie skórki:

$$\left. -k \frac{\partial T}{\partial z} \right|_{z=0}
 = h_c \left( T_{\text{pan,in}} - T|_{z=0} \right) - \dot m_{\downarrow}\, h_{fg}(T) .$$

**Warunek brzegowy na górze** (powierzchnia odkryta) — konwekcja swobodna, promieniowanie
i parowanie do otoczenia:

$$\left. -k \frac{\partial T}{\partial z} \right|_{z=L}
 = -\Big[ h_{\text{top}} \left( T|_{z=L} - T_\infty \right)
 + \varepsilon \sigma \big( T|_{z=L}^4 - T_\infty^4 \big)
 + \dot m_{\uparrow} h_{fg}(T) \Big] .$$

**Strumień pary z powierzchni.** Poniżej 100 °C parowanie jest ograniczone dyfuzją pary do
powietrza (analogia Chiltona–Colburna), powyżej 100 °C — transportem wody kapilarami przez
tworzącą się skórkę:

$$\dot m = f_W \Big[ \dot m_{\max} S(T)
 + h_m \big( a_w \rho_{\text{sat}}(T) - \varphi\, \rho_{\text{sat}}(T_\infty) \big) \big(1 - S(T)\big) \Big],
\qquad S(T) = \frac{1}{1 + e^{-(T - 100)/4}},$$

gdzie $h_m = h / (\rho_a c_{p,a} \mathrm{Le}^{2/3})$, a $f_W \in [0,1]$ to ułamek wody wolnej,
która jeszcze została za daną powierzchnią. Ten człon jest odpowiedzialny za plateau przy
około 100 °C i za to, że brązowienie zaczyna się dopiero po wysuszeniu powierzchni.

**Patelnia** — dwa węzły skupione: krążek pod kotletem ($C_{\text{in}}$) i reszta dna
razem ze ścianką ($C_{\text{out}}$), sprzężone przewodzeniem promieniowym w płycie dna:

$$C_{\text{in}} \frac{dT_{\text{in}}}{dt} = f P(t) - A_m q_{\text{kontakt}} - G\,(T_{\text{in}} - T_{\text{out}}),$$
$$C_{\text{out}} \frac{dT_{\text{out}}}{dt} = (1-f) P(t) + G\,(T_{\text{in}} - T_{\text{out}}) - Q_{\text{straty}},
\qquad G = \frac{2 \pi k_{\text{pan}} \delta}{\ln (r_2 / r_1)} .$$

**Źródło ciepła** — moc użyteczna z bezwładnością pierwszego rzędu:

$$\frac{dP}{dt} = \frac{P_{\text{set}} - P}{\tau}, \qquad
P_{\text{set}} = s \cdot P_{\text{nom}} \cdot \eta_{\text{dost}} \cdot f_{\text{geo}} .$$

**Bezpieczeństwo mikrobiologiczne** — skumulowana letalność w najzimniejszym punkcie,
w jednostkach redukcji dziesiętnych:

$$\frac{d\Lambda}{dt} = \frac{1}{D_{\text{ref}}}\, 10^{(T_{\min} - T_{\text{ref}})/z} .$$

### 1.3. Czego model nie obejmuje

- Kurczenia się mięsa i wyciskania soku przez denaturację kolagenu (ubytek masy jest więc
  zaniżony — model liczy tylko parowanie).
- Ruchu oleju, wrzenia oleju i przenoszenia ciepła przez warstwę oleju jako osobne medium;
  olej jest ukryty w wartości $h_c$.
- Dwuwymiarowego pola w mięsie (brzegi kotleta gotują się w rzeczywistości szybciej).
- Właściwości termofizycznych powyżej 150 °C: korelacje Choi–Okos są tam ekstrapolacją,
  więc temperatura podawana do nich jest przycinana do zakresu ważności.
- Kinetyki denaturacji białek jako osobnego pochłaniacza energii (mała w porównaniu z parowaniem).

## 2. Konfiguracja i zależności

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

DATA_DIR = Path.cwd()
SIGMA = 5.670374419e-8          # Stefan-Boltzmann constant, W/(m2 K4)

## Categorical palette in fixed assignment order (Okabe-Ito subset, validated for CVD
## separation). Series identity is always carried by a legend entry as well as by colour.
SERIES_COLORS = ["#0072B2", "#D55E00", "#009E73", "#56B4E9", "#CC79A7", "#E69F00"]
SEQUENTIAL_CMAP = "magma"       # single-hue-family, perceptually uniform, for temperature fields

mpl.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (8.0, 4.5),
    "axes.prop_cycle": mpl.cycler(color=SERIES_COLORS),
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "grid.color": "#dcdcdc",
    "grid.linewidth": 0.6,
    "lines.linewidth": 2.0,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "font.size": 9,
})

## 3. Właściwości termofizyczne mięsa — model Choi–Okos

Zamiast pojedynczych tabelarycznych wartości $k$, $\rho$, $c_p$ dla „wołowiny” używamy modelu
składnikowego Choi i Okos (1986): właściwości każdego czystego składnika (woda, białko,
tłuszcz, węglowodany, błonnik, popiół) są wielomianami drugiego stopnia względem temperatury,
a właściwość mięsa jest ich mieszaniną. Dzięki temu jedno źródło danych — skład podstawowy —
daje spójne $k(T)$, $\rho(T)$ i $c_p(T)$ dla dowolnego kawałka mięsa i ryby.

Reguły mieszania: $c_p$ ważone masowo, $\rho$ jako odwrotność sumy $x_i/\rho_i$, $k$ ważone
objętościowo (model równoległy). Model równoległy zawyża $k$ dla włókien prostopadłych do
strumienia o około 10 %; dla mięs różnica $k_\parallel$ vs $k_\perp$ w danych pomiarowych
ASHRAE wynosi kilka procent, więc pozostajemy przy modelu równoległym.

Współczynniki zostały przeliczone do SI dokładnym podstawieniem $t_F = 1{,}8\,T_C + 32$ do
wielomianów w jednostkach I-P z tablic 1 i 2 rozdziału „Thermal Properties of Foods”
*ASHRAE Handbook — Refrigeration*. Zakres ważności: $-40\ldots150$ °C.

In [ ]:
_COEF = pd.read_csv(DATA_DIR / "heat_capacities/component_properties_choi_okos.csv", comment="#")
COMPONENTS = ("water", "protein", "fat", "carbohydrate", "fiber", "ash")
T_VALID = (-40.0, 150.0)

_POLY = {
    (p, c): _COEF[(_COEF["property"] == p) & (_COEF["component"] == c)][["a", "b", "c"]]
            .to_numpy()[0]
    for p in ("k", "rho", "cp") for c in COMPONENTS
}


def component_property(prop: str, comp: str, T):
    """Pure-component property at temperature T.

    :param prop: one of ``"k"`` (W/(m K)), ``"rho"`` (kg/m3), ``"cp"`` (J/(kg K)).
    :param comp: component name from :data:`COMPONENTS`.
    :param T: temperature in degrees Celsius; clipped to the validity range of the correlation.
    :returns: property value, same shape as ``T``.
    """
    a, b, c = _POLY[(prop, comp)]
    Tc = np.clip(T, *T_VALID)
    return a + b * Tc + c * Tc**2


@dataclass(frozen=True)
class Composition:
    """Proximate composition of a food, as mass fractions summing to one."""

    water: float
    protein: float
    fat: float
    carbohydrate: float
    fiber: float
    ash: float
    free_water_fraction: float = 0.35
    a_w: float = 0.99
    name: str = ""

    @classmethod
    def from_row(cls, row: pd.Series) -> "Composition":
        """Build a composition from one row of ``food.csv``, renormalising to 100 %."""
        keys = ("water", "protein", "fat", "carb", "fiber", "ash")
        x = np.array([row[f"{k}_pct"] for k in keys], dtype=float)
        x /= x.sum()
        return cls(*x, float(row["water_frac_free"]), float(row["a_w"]), str(row["name_en"]))

    @property
    def fractions(self) -> dict[str, float]:
        return dict(zip(COMPONENTS, (self.water, self.protein, self.fat,
                                     self.carbohydrate, self.fiber, self.ash)))

    def rho(self, T):
        """Density, kg/m3 (inverse additive volume mixing rule)."""
        return 1.0 / sum(x / component_property("rho", c, T) for c, x in self.fractions.items())

    def cp(self, T):
        """Specific heat capacity, J/(kg K) (mass-weighted mixing rule)."""
        return sum(x * component_property("cp", c, T) for c, x in self.fractions.items())

    def k(self, T):
        """Thermal conductivity, W/(m K) (volume-weighted parallel model)."""
        rho = self.rho(T)
        return sum(x * rho / component_property("rho", c, T) * component_property("k", c, T)
                   for c, x in self.fractions.items())

    def alpha(self, T):
        """Thermal diffusivity, m2/s."""
        return self.k(T) / (self.rho(T) * self.cp(T))

In [ ]:
def h_fg(T):
    """Latent heat of vaporisation of water, J/kg, for 0-150 degrees Celsius."""
    return 2.501e6 - 2369.0 * np.clip(T, 0.0, 150.0)


def p_sat(T):
    """Saturation vapour pressure of water over a flat surface, Pa (Tetens equation)."""
    return 610.78 * np.exp(17.2694 * T / (T + 238.3))


def rho_vap_sat(T):
    """Saturated water-vapour concentration, kg/m3, from the ideal-gas law."""
    return p_sat(T) * 0.018015 / (8.3145 * (T + 273.15))

## 4. Wczytanie danych materiałowych

In [ ]:
FOODS = pd.read_csv(DATA_DIR / "heat_capacities/food.csv", comment="#").set_index("food_id")
PANS = pd.read_csv(DATA_DIR / "heat_capacities/pan_capacities.csv",
                   comment="#").set_index("material_id")
SOURCES = pd.read_csv(DATA_DIR / "data/heat_sources.csv", comment="#").set_index("source_id")
LETHALITY_TABLE = pd.read_csv(DATA_DIR / "data/usda_appendix_a_lethality.csv", comment="#")

## Derived thermophysical properties at 20 C, for orientation
summary = pd.DataFrame(
    [{
        "name_pl": row["name_pl"],
        "water_%": row["water_pct"],
        "fat_%": row["fat_pct"],
        "rho_20C": Composition.from_row(row).rho(20.0),
        "cp_20C": Composition.from_row(row).cp(20.0),
        "k_20C": Composition.from_row(row).k(20.0),
        "alpha_20C_1e7": Composition.from_row(row).alpha(20.0) * 1e7,
    } for _, row in FOODS.iterrows()],
    index=FOODS.index,
)
summary.round(3)

In [ ]:
PANS[["name_pl", "k_W_mK", "rho_kg_m3", "cp_J_kgK", "emissivity_outside",
      "thickness_mm_default", "ferromagnetic"]]

In [ ]:
SOURCES[["name_pl", "coupling", "power_nominal_W", "eff_heatup", "eff_saute",
         "t_element_s", "T_source_C"]]

Pierwszy wykres pokazuje, skąd bierze się zróżnicowanie mięs. Dyfuzyjność cieplna
$\alpha = k/(\rho c_p)$ decyduje o tempie nagrzewania wnętrza i jest zdominowana przez
zawartość wody i tłuszczu. Tłuszcz ma około czterokrotnie niższe $k$ niż woda, a obniża
$\rho c_p$ słabiej, niż obniża $k$ — w efekcie tłuste kawałki mają *niższą* dyfuzyjność
i nagrzewają się w środku wolniej. W tabeli powyżej boczek ma
$\alpha \approx 1{,}11 \cdot 10^{-7}$ m²/s wobec $1{,}34 \cdot 10^{-7}$ dla schabu, czyli
różnica około 20 %. Jednocześnie tłusty kawałek ma dużo mniej wody do odparowania, więc szybciej
wychodzi z plateau 100 °C i szybciej się rumieni. Te dwa efekty działają przeciwnie i model
rozstrzyga, który wygrywa dla konkretnej geometrii.

In [ ]:
## Property curves for a representative subset of the dataset
selection = ["pork_loin_lean", "beef_sirloin_lean", "chicken_breast", "salmon_pink", "pork_belly"]
T_grid = np.linspace(0.0, 150.0, 200)

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6))
for food_id, color in zip(selection, SERIES_COLORS):
    comp = Composition.from_row(FOODS.loc[food_id])
    label = FOODS.loc[food_id, "name_pl"]
    axes[0].plot(T_grid, comp.k(T_grid), color=color, label=label)
    axes[1].plot(T_grid, comp.cp(T_grid) / 1000.0, color=color, label=label)
    axes[2].plot(T_grid, comp.alpha(T_grid) * 1e7, color=color, label=label)

for ax, ylabel, title in zip(
        axes,
        ["k  [W/(m K)]", "c_p  [kJ/(kg K)]", "alpha  [1e-7 m2/s]"],
        ["Thermal conductivity", "Specific heat capacity", "Thermal diffusivity"]):
    ax.set_xlabel("Temperature  [deg C]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
axes[0].legend(loc="lower right")
fig.suptitle("Choi-Okos properties of raw meats (composition-derived)", y=1.02)
fig.tight_layout()
plt.show()

## 5. Model źródła ciepła

Trzy sposoby grzania różnią się nie tylko sprawnością, ale i mechanizmem sprzężenia z patelnią,
co daje inną wrażliwość na geometrię i inną bezwładność.

| Sprzężenie | Mechanizm | Na co reaguje odległość |
|---|---|---|
| **indukcja** | prądy wirowe wzbudzane bezpośrednio w ferromagnetycznym dnie | brak — liczy się pokrycie cewki dnem patelni |
| **płyta promiennikowa / spirala** | promieniowanie i przewodzenie przez szczelinę powietrzną | szczelina dno–płyta (wypaczone dno = spadek mocy) |
| **palnik gazowy** | strumień gorących spalin uderzający w dno, plus promieniowanie płomienia | odległość palnik–dno: wraz z nią rośnie promień strugi i część spalin omija patelnię |

**Sprawność.** Punktem wyjścia są zmierzone sprawności przekazu energii z badania
porównawczego płyt domowych (test nagrzewania wody, ASTM F1275): indukcja 83–86 %, płyta
ceramiczna 75,5 %, spirala elektryczna 79,3 %, palnik gazowy 31,9 %. Te liczby obejmują
*również* straty z naczynia do pomieszczenia, a straty patelni model liczy osobno. Dlatego
sprawność dostarczania definiujemy jako

$$\eta_{\text{dost}} = \min\!\left(0{,}95,\ \frac{\eta_{\text{zmierzona}}}{\eta_{\text{naczynia}}}\right),
\qquad \eta_{\text{naczynia}} \approx 0{,}90,$$

gdzie $\eta_{\text{naczynia}}$ to przyjęty ułamek energii wchodzącej do naczynia, który trafił
do wody w teście odniesienia. To jest jawne założenie modelu, nie wielkość zmierzona.

**Czynnik geometryczny** $f_{\text{geo}}$ jest znormalizowany do warunków odniesienia
(szczelina 0,3 mm dla płyty, odległość 20 mm dla gazu), więc przy warunkach odniesienia model
odtwarza zmierzoną sprawność dokładnie. Dla płyty promiennikowej przewodność szczeliny to

$$h_{\text{gap}} = \frac{k_{\text{pow}}}{d}
 + \frac{\sigma (T_e^2 + T_p^2)(T_e + T_p)}{1/\varepsilon_e + 1/\varepsilon_p - 1},$$

a dla palnika gazowego przechwycona część strugi to $\min\!\big(1, (R_{\text{pan}}/R_{\text{strugi}})^2\big)$
przy $R_{\text{strugi}} = R_{\text{palnika}} + d \tan\theta$, $\theta = 15°$.

Są to korelacje inżynierskie kalibrowane do danych pomiarowych, a nie zwalidowane modele
spalania — służą do porównywania wariantów, nie do przewidywania sprawności w wartościach
bezwzględnych poza punktem kalibracji.

In [ ]:
@dataclass
class Geometry:
    """Dimensions of the cutlet and of the pan, in metres."""

    thickness_m: float = 0.015
    radius_m: float = 0.045
    pan_radius_m: float = 0.12
    pan_thickness_m: float = 0.0045
    pan_wall_height_m: float = 0.045
    coil_radius_m: float = 0.09

    @property
    def area_meat(self):
        return np.pi * self.radius_m**2

    @property
    def area_pan(self):
        return np.pi * self.pan_radius_m**2

    @property
    def area_wall(self):
        return 2 * np.pi * self.pan_radius_m * self.pan_wall_height_m


@dataclass
class PanMaterial:
    """Thermal properties of the pan base material."""

    k: float
    rho: float
    cp: float
    emissivity: float
    name: str = ""

    @classmethod
    def from_row(cls, row: pd.Series) -> "PanMaterial":
        return cls(float(row["k_W_mK"]), float(row["rho_kg_m3"]), float(row["cp_J_kgK"]),
                   float(row["emissivity_outside"]), str(row["name_pl"]))


@dataclass
class Environment:
    """Kitchen conditions and the external transfer coefficients."""

    T_air_C: float = 22.0
    rh: float = 0.45
    h_top: float = 12.0            # free convection above the cutlet
    h_pan_free: float = 9.0        # free convection from the exposed pan surfaces
    emissivity_food: float = 0.92
    UA_handle: float = 0.35        # lumped conduction loss through handle and rim


@dataclass
class Contact:
    """Thermal contact between the pan and the cutlet."""

    h_contact: float = 900.0       # W/(m2 K); Dagerskog (1985) measured 200-1200


@dataclass
class NumericalSetup:
    """Discretisation and the evaporation-model constants."""

    n_nodes: int = 41
    m_evap_max: float = 1.0e-2     # kg/(m2 s): capillary transport limit through the crust
    T_boil_C: float = 100.0
    boil_width_K: float = 4.0

In [ ]:
@dataclass
class HeatSource:
    """A cooktop burner or hob, with its coupling mechanism to the pan base."""

    coupling: str                  # "induction" | "radiant" | "flame"
    power_W: float
    eff_measured: float
    tau_s: float
    T_source_C: float
    name: str = ""
    vessel_eff_ref: float = 0.90   # assumed pot-to-room efficiency in the reference test
    gap_m: float = 0.0003          # radiant hobs: pan base to hob surface
    distance_m: float = 0.020      # gas burners: burner head to pan base
    burner_radius_m: float = 0.045
    plume_half_angle_deg: float = 15.0
    spread_factor: float = 1.6     # radial spreading of the impinging flue gas

    REF_GAP_M = 3e-4
    REF_DISTANCE_M = 0.020

    @classmethod
    def from_row(cls, row: pd.Series, **kwargs) -> "HeatSource":
        return cls(str(row["coupling"]), float(row["power_nominal_W"]), float(row["eff_heatup"]),
                   float(row["t_element_s"]), float(row["T_source_C"]), str(row["name_pl"]),
                   **kwargs)

    @property
    def eff_delivery(self) -> float:
        """Fraction of the input energy that reaches the pan body."""
        return min(0.95, self.eff_measured / self.vessel_eff_ref)

    # --- geometric coupling ----------------------------------------------------------
    def _gap_conductance(self, d: float, pan: PanMaterial) -> float:
        """Conduction plus radiation across the hob-to-pan gap, W/(m2 K)."""
        k_air = 0.055                                   # air at a few hundred degrees C
        T_e, T_p = self.T_source_C + 273.15, 773.15     # emitter and pan base, K
        h_rad = (SIGMA * (T_e**2 + T_p**2) * (T_e + T_p)
                 / (1.0 / 0.9 + 1.0 / pan.emissivity - 1.0))
        return k_air / max(d, 5e-5) + h_rad

    def _intercepted_fraction(self, d: float, geom: Geometry) -> float:
        """Fraction of the flame plume cross-section caught by the pan base."""
        r_plume = self.burner_radius_m + d * np.tan(np.radians(self.plume_half_angle_deg))
        return float(min(1.0, (geom.pan_radius_m / r_plume) ** 2))

    def geometric_factor(self, geom: Geometry, pan: PanMaterial) -> float:
        """Coupling relative to the reference conditions at which the efficiency was measured."""
        if self.coupling == "induction":
            return float(min(1.0, geom.area_pan / (np.pi * geom.coil_radius_m**2)))
        if self.coupling == "radiant":
            return self._gap_conductance(self.gap_m, pan) / self._gap_conductance(self.REF_GAP_M, pan)
        if self.coupling == "flame":
            return (self._intercepted_fraction(self.distance_m, geom)
                    / self._intercepted_fraction(self.REF_DISTANCE_M, geom))
        raise ValueError(f"unknown coupling: {self.coupling}")

    def heated_radius(self, geom: Geometry) -> float:
        """Radius of the pan base over which the delivered power is taken to be uniform."""
        if self.coupling == "flame":
            r_plume = (self.burner_radius_m
                       + self.distance_m * np.tan(np.radians(self.plume_half_angle_deg)))
            r = self.spread_factor * r_plume
        else:
            r = geom.coil_radius_m
        return float(min(geom.pan_radius_m, r))

    def split(self, geom: Geometry) -> float:
        """Fraction of the delivered power landing on the pan area under the cutlet."""
        a_heated = np.pi * self.heated_radius(geom) ** 2
        return float(min(geom.area_meat, a_heated) / a_heated)

    def useful_power(self, setting: float, geom: Geometry, pan: PanMaterial) -> float:
        """Power actually entering the pan body at a given knob setting, W."""
        return setting * self.power_W * self.eff_delivery * self.geometric_factor(geom, pan)

### 5.1. Wpływ odległości / szczeliny na moc dostarczaną

Wykres pokazuje, dlaczego „odległość od patelni” znaczy co innego przy każdym źródle: płyta
promiennikowa traci moc gwałtownie już przy szczelinie ułamka milimetra (wypaczone dno),
podczas gdy palnik gazowy degraduje się łagodnie, aż struga spalin przestanie mieścić się
pod patelnią. Indukcja jest od tej zmiennej niezależna.

In [ ]:
geom_ref = Geometry()
pan_ref = PanMaterial.from_row(PANS.loc["cast_iron"])

gap_mm = np.linspace(0.05, 4.0, 200)
dist_mm = np.linspace(2.0, 60.0, 200)

radiant = HeatSource.from_row(SOURCES.loc["electric_ceramic_2500"])
flame = HeatSource.from_row(SOURCES.loc["gas_burner_4980"])
induction = HeatSource.from_row(SOURCES.loc["induction_3600"])

eff_radiant = [HeatSource.from_row(SOURCES.loc["electric_ceramic_2500"], gap_m=g * 1e-3)
               .eff_delivery
               * HeatSource.from_row(SOURCES.loc["electric_ceramic_2500"], gap_m=g * 1e-3)
               .geometric_factor(geom_ref, pan_ref) for g in gap_mm]
eff_flame = [HeatSource.from_row(SOURCES.loc["gas_burner_4980"], distance_m=d * 1e-3).eff_delivery
             * HeatSource.from_row(SOURCES.loc["gas_burner_4980"], distance_m=d * 1e-3)
             .geometric_factor(geom_ref, pan_ref) for d in dist_mm]
eff_induction = induction.eff_delivery * induction.geometric_factor(geom_ref, pan_ref)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharey=True)
axes[0].plot(gap_mm, 100 * np.array(eff_radiant), color=SERIES_COLORS[0],
             label="Radiant glass-ceramic hob")
axes[0].axhline(100 * eff_induction, color=SERIES_COLORS[2], ls="--", label="Induction (no gap)")
axes[0].set_xlabel("Pan-to-hob gap  [mm]")
axes[0].set_title("Electric hob: effect of a warped base")
axes[1].plot(dist_mm, 100 * np.array(eff_flame), color=SERIES_COLORS[1], label="Gas burner")
axes[1].set_xlabel("Burner-to-pan distance  [mm]")
axes[1].set_title("Gas burner: effect of the flame stand-off")
for ax in axes:
    ax.set_ylabel("Delivery efficiency to the pan  [%]")
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 6. Kinetyka inaktywacji mikrobiologicznej

Bezpieczeństwo nie jest progiem temperatury, tylko całką czas–temperatura. Do tabel USDA FSIS
Appendix A (czas utrzymania w danej temperaturze potrzebny do redukcji *Salmonelli* o zadaną
liczbę rzędów) dopasowujemy klasyczne parametry $D$ i $z$ przez regresję liniową
$\log_{10} t$ względem $T$. Dopasowane $z \approx 5{,}5$ °C jest zgodne z wartościami
literaturowymi dla *Salmonelli* w mięsie.

In [ ]:
@dataclass(frozen=True)
class Lethality:
    """First-order thermal death kinetics fitted to a USDA Appendix A column."""

    D_ref_s: float
    z_C: float
    T_ref_C: float
    n_log: float
    label: str

    @classmethod
    def fit(cls, table: pd.DataFrame, column: str, n_log: float,
            T_ref_C: float = 60.0) -> "Lethality":
        """Fit D and z to one column of the time-temperature table.

        :param table: the Appendix A table with a ``T_C`` column.
        :param column: the column holding the required holding time in seconds.
        :param n_log: number of log10 reductions that column corresponds to.
        :returns: fitted kinetics referenced at ``T_ref_C``.
        """
        sub = table[["T_C", column]].dropna()
        sub = sub[sub[column] > 0]
        slope, intercept = np.polyfit(sub["T_C"], np.log10(sub[column]), 1)
        t_ref = 10 ** (intercept + slope * T_ref_C)
        return cls(t_ref / n_log, -1.0 / slope, T_ref_C, n_log,
                   f"{column}: z = {-1.0 / slope:.2f} K, D60 = {t_ref / n_log:.0f} s")

    def rate(self, T):
        """Log10 reductions accumulated per second at temperature T."""
        return 10 ** ((T - self.T_ref_C) / self.z_C) / self.D_ref_s


LETHALITY = {
    "beef_6_5log": Lethality.fit(LETHALITY_TABLE, "beef_6_5log_s", 6.5),
    "beef_7log": Lethality.fit(LETHALITY_TABLE, "beef_7log_s", 7.0),
    "chicken_7log": Lethality.fit(LETHALITY_TABLE, "chicken_7log_s", 7.0),
    "turkey_7log": Lethality.fit(LETHALITY_TABLE, "turkey_7log_s", 7.0),
}
for key, model in LETHALITY.items():
    print(f"{key:16s} {model.label}")

## 7. Solver

Dyskretyzacja objętości skończonych: $n$ węzłów o jednakowej grubości $\Delta z = L/n$,
przewodność na ściance liczona jako średnia harmoniczna sąsiadów, opór kontaktowy złożony
szeregowo z połową komórki brzegowej. Całkowanie w czasie metodą LSODA (automatyczne
przełączanie sztywny/niesztywny), bo po zetknięciu z patelnią układ jest silnie sztywny:
stałe czasowe węzła powierzchniowego i rdzenia różnią się o trzy rzędy wielkości.

Zmienne stanu: moc dostarczana $P$, dwie temperatury patelni, $n$ temperatur mięsa, dwa zapasy
wody wolnej (po jednym na powierzchnię), skumulowana letalność i dwa wskaźniki brązowienia.
Obrócenie kotleta realizowane jest jako przerwanie całkowania, odwrócenie profilu temperatury
i zamiana stanów obu powierzchni.

In [ ]:
@dataclass
class FryingSetup:
    """Everything needed to define one frying scenario."""

    comp: Composition
    pan: PanMaterial
    source: HeatSource
    geom: Geometry = field(default_factory=Geometry)
    env: Environment = field(default_factory=Environment)
    contact: Contact = field(default_factory=Contact)
    num: NumericalSetup = field(default_factory=NumericalSetup)
    lethality: "Lethality | None" = None
    power_setting: float = 1.0


class FryingModel:
    """One-dimensional conduction model of a cutlet resting on a two-node pan."""

    N_EXTRA = 8   # P, T_pan_in, T_pan_out, W_down, W_up, lethality, browning_down, browning_up

    def __init__(self, s: FryingSetup):
        self.s = s
        n = s.num.n_nodes
        self.n = n
        self.dz = s.geom.thickness_m / n
        self.z = (np.arange(n) + 0.5) * self.dz

        ## Pan nodes: a disc under the cutlet and the surrounding annulus
        A_m, A_p = s.geom.area_meat, s.geom.area_pan
        cap = s.pan.rho * s.pan.cp * s.geom.pan_thickness_m
        self.C_pan_in = cap * A_m
        self.C_pan_out = cap * (A_p - A_m) + cap * s.geom.area_wall

        ### REMARK: Radial conduction between the two pan nodes, taken between the area centroid
        ### of the inner disc (R_m/2) and that of the outer annulus. This is the term that makes
        ### pan material matter: it sets how fast the cold spot under the cutlet recovers.
        r1 = 0.5 * s.geom.radius_m
        r2 = np.sqrt(0.5 * (s.geom.radius_m**2 + s.geom.pan_radius_m**2))
        self.G_lat = 2 * np.pi * s.pan.k * s.geom.pan_thickness_m / np.log(r2 / r1)

        self.f_inner = s.source.split(s.geom)
        self.P_useful = s.source.useful_power(s.power_setting, s.geom, s.pan)

        ## Free water available to each face, per unit area of that face
        self.W_face0 = (s.comp.water * s.comp.free_water_fraction
                        * s.comp.rho(20.0) * 0.5 * s.geom.thickness_m)
        self.mass = s.comp.rho(20.0) * s.geom.thickness_m * A_m

    # --- state packing ---------------------------------------------------------------
    def unpack(self, y):
        n = self.n
        return dict(P=y[0], Tin=y[1], Tout=y[2], T=y[3:3 + n], W_dn=y[3 + n], W_up=y[4 + n],
                    L=y[5 + n], B_dn=y[6 + n], B_up=y[7 + n])

    def initial_state(self, T_meat0, T_pan0):
        y = np.zeros(self.N_EXTRA + self.n + 1)
        y[0] = self.P_useful
        y[1] = y[2] = T_pan0
        y[3:3 + self.n] = T_meat0
        y[3 + self.n] = y[4 + self.n] = self.W_face0
        return y

    # --- constitutive relations ------------------------------------------------------
    def boiling_switch(self, T):
        """Smooth 0->1 switch around the boiling point of the tissue water."""
        return 1.0 / (1.0 + np.exp(-(T - self.s.num.T_boil_C) / self.s.num.boil_width_K))

    def evaporation_flux(self, T_surf, W, exposed):
        """Water flux leaving one face, kg/(m2 s).

        Below the boiling point the face dries by vapour diffusion into the room; above it the
        flux is limited by capillary transport of water through the forming crust. Both branches
        are scaled by the fraction of free water still present behind that face.
        """
        s = self.s
        f_W = np.clip(W / self.W_face0, 0.0, 1.0)
        S = self.boiling_switch(T_surf)
        m_boil = s.num.m_evap_max * S
        if exposed:
            h_m = s.env.h_top / (1.2 * 1006.0 * 0.85 ** (2 / 3))   # Chilton-Colburn analogy
            drive = (s.comp.a_w * rho_vap_sat(np.clip(T_surf, 0.0, 99.0))
                     - s.env.rh * rho_vap_sat(s.env.T_air_C))
            m_dry = h_m * max(float(drive), 0.0) * (1.0 - S)
        else:
            m_dry = 0.0
        return f_W * (m_boil + m_dry)

    # --- right-hand side -------------------------------------------------------------
    def rhs(self, t, y, meat_present=True):
        s, n = self.s, self.n
        st = self.unpack(y)
        T = np.clip(st["T"], -20.0, 400.0)
        Tin, Tout, P = st["Tin"], st["Tout"], st["P"]
        T_air = s.env.T_air_C
        A_m, A_p = s.geom.area_meat, s.geom.area_pan
        dy = np.zeros_like(y)

        # Heat source: first-order lag between the setting and the power actually delivered
        dy[0] = (self.P_useful - P) / max(s.source.tau_s, 1e-3)

        def surface_loss(T_s, area, emissivity, h_conv):
            return area * (h_conv * (T_s - T_air)
                           + emissivity * SIGMA * ((T_s + 273.15) ** 4 - (T_air + 273.15) ** 4))

        # Meat: finite-volume conduction with evaporative sinks on both faces
        if meat_present:
            k = s.comp.k(T)
            k_face = 2 * k[:-1] * k[1:] / (k[:-1] + k[1:])
            q_int = -k_face * np.diff(T) / self.dz                   # W/m2, positive upwards

            ## Bottom face: contact conductance in series with half a control volume
            U_bot = 1.0 / (1.0 / s.contact.h_contact + 0.5 * self.dz / k[0])
            m_dn = self.evaporation_flux(T[0], st["W_dn"], exposed=False)
            q_bot = U_bot * (Tin - T[0]) - m_dn * h_fg(T[0])

            ## Top face: convection, radiation and evaporation into the room
            U_top = 1.0 / (1.0 / s.env.h_top + 0.5 * self.dz / k[-1])
            m_up = self.evaporation_flux(T[-1], st["W_up"], exposed=True)
            q_top = (U_top * (T[-1] - T_air)
                     + s.env.emissivity_food * SIGMA
                     * ((T[-1] + 273.15) ** 4 - (T_air + 273.15) ** 4)
                     + m_up * h_fg(T[-1]))

            q = np.concatenate(([q_bot], q_int, [q_top]))            # (n+1,) face fluxes
            cap = s.comp.rho(T) * s.comp.cp(T) * self.dz             # J/(m2 K) per node
            dy[3:3 + n] = (q[:-1] - q[1:]) / cap

            dy[3 + n] = -m_dn
            dy[4 + n] = -m_up
            if s.lethality is not None:
                dy[5 + n] = s.lethality.rate(T.min())
            dy[6 + n] = browning_rate(T[0], st["W_dn"] / self.W_face0)
            dy[7 + n] = browning_rate(T[-1], st["W_up"] / self.W_face0)

            q_to_meat = U_bot * (Tin - T[0]) * A_m
            loss_inner = 0.0
        else:
            q_to_meat = 0.0
            loss_inner = surface_loss(Tin, A_m, s.pan.emissivity, s.env.h_pan_free)

        # Pan: inner disc, outer annulus plus wall, coupled by radial conduction
        G = self.G_lat
        dy[1] = (P * self.f_inner - q_to_meat - G * (Tin - Tout) - loss_inner) / self.C_pan_in
        loss_outer = (surface_loss(Tout, A_p - A_m, s.pan.emissivity, s.env.h_pan_free)
                      + surface_loss(Tout, s.geom.area_wall, s.pan.emissivity, s.env.h_pan_free)
                      + s.env.UA_handle * (Tout - T_air))
        dy[2] = (P * (1 - self.f_inner) + G * (Tin - Tout) - loss_outer) / self.C_pan_out
        return dy

    # --- drivers ---------------------------------------------------------------------
    def preheat(self, t_end=900.0, T_pan0=None, dt_out=2.0):
        """Empty-pan warm-up from ambient at the configured power setting."""
        T0 = self.s.env.T_air_C if T_pan0 is None else T_pan0
        y = self.initial_state(T0, T0)
        y[0] = 0.0
        t_eval = np.arange(0.0, t_end + 1e-9, dt_out)
        sol = solve_ivp(self.rhs, (0.0, t_end), y, args=(False,), method="LSODA",
                        t_eval=t_eval, rtol=1e-6, atol=1e-8, max_step=10.0)
        return SimulationResult(self, sol.t, sol.y)

    def steady_pan_temperature(self, setting):
        """Equilibrium temperature of an empty pan at a given power setting, deg C."""
        s = self.s
        P = s.source.useful_power(setting, s.geom, s.pan)
        area = s.geom.area_pan + s.geom.area_wall
        T_air = s.env.T_air_C

        def residual(T):
            loss = area * (s.env.h_pan_free * (T - T_air)
                           + s.pan.emissivity * SIGMA
                           * ((T + 273.15) ** 4 - (T_air + 273.15) ** 4))
            return P - loss - s.env.UA_handle * (T - T_air)

        lo, hi = T_air, 1200.0
        for _ in range(200):
            mid = 0.5 * (lo + hi)
            if residual(mid) > 0:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)

    def setting_for_pan_temperature(self, T_target):
        """Power setting whose empty-pan equilibrium is T_target (clipped to [0, 1])."""
        lo, hi = 0.0, 1.0
        if self.steady_pan_temperature(1.0) < T_target:
            return 1.0
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if self.steady_pan_temperature(mid) < T_target:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)

    def run(self, t_end, flips=(), T_meat0=6.0, T_pan0=200.0, dt_out=1.0):
        """Fry for t_end seconds, turning the cutlet at each time in flips."""
        y = self.initial_state(T_meat0, T_pan0)
        edges = [0.0, *sorted(flips), t_end]
        ts, ys, n = [], [], self.n
        for a, b in zip(edges[:-1], edges[1:]):
            if b <= a:
                continue
            t_eval = np.arange(a, b + 1e-9, dt_out)
            sol = solve_ivp(self.rhs, (a, b), y, args=(True,), method="LSODA",
                            t_eval=t_eval, rtol=1e-6, atol=1e-8, max_step=2.0)
            if not sol.success:
                raise RuntimeError(sol.message)
            ts.append(sol.t)
            ys.append(sol.y)
            y = sol.y[:, -1].copy()
            ## Turning the cutlet: mirror the profile and exchange the two faces
            y[3:3 + n] = y[3:3 + n][::-1]
            y[3 + n], y[4 + n] = y[4 + n], y[3 + n]
            y[6 + n], y[7 + n] = y[7 + n], y[6 + n]
        return SimulationResult(self, np.concatenate(ts), np.concatenate(ys, axis=1))


def browning_rate(T, moisture_fraction=0.0):
    """Relative Maillard rate on a face.

    Normalised so that one minute at 160 C on a dry surface gives an index of 1. Browning is
    suppressed while the surface still carries free water, because the surface then cannot get
    meaningfully above 100 C and the reaction is water-inhibited.
    """
    Ea, R = 120e3, 8.314
    ref = np.exp(-Ea / (R * (160.0 + 273.15)))
    dry = 1.0 - np.clip(moisture_fraction, 0.0, 1.0)
    return float(np.exp(-Ea / (R * (np.clip(T, -20, 400) + 273.15))) / ref / 60.0 * dry)


@dataclass
class SimulationResult:
    model: FryingModel
    t: np.ndarray
    Y: np.ndarray

    @property
    def n(self):
        return self.model.n

    @property
    def T(self):
        return self.Y[3:3 + self.n]

    @property
    def T_core(self):
        return self.T.min(axis=0)

    @property
    def T_centre(self):
        return self.T[self.n // 2]

    @property
    def T_bottom(self):
        return self.T[0]

    @property
    def T_top(self):
        return self.T[-1]

    @property
    def T_pan_in(self):
        return self.Y[1]

    @property
    def T_pan_out(self):
        return self.Y[2]

    @property
    def lethality(self):
        return self.Y[5 + self.n]

    @property
    def browning(self):
        return np.maximum(self.Y[6 + self.n], self.Y[7 + self.n])

    @property
    def water_loss_fraction(self):
        m = self.model
        lost = 2 * m.W_face0 - self.Y[3 + self.n] - self.Y[4 + self.n]
        return lost * m.s.geom.area_meat / m.mass

    def time_to_core(self, T_target):
        reached = np.flatnonzero(self.T_core >= T_target)
        return float(self.t[reached[0]]) if reached.size else float("nan")

In [ ]:
def build_model(food_id: str, pan_id: str, source_id: str,
                lethality_key: str = "beef_6_5log", **kwargs) -> FryingModel:
    """Assemble a :class:`FryingModel` from the dataset identifiers.

    :param food_id: row of ``food.csv``.
    :param pan_id: row of ``pan_capacities.csv``.
    :param source_id: row of ``heat_sources.csv``.
    :param lethality_key: key into :data:`LETHALITY`.
    :param kwargs: forwarded to :class:`FryingSetup` (``geom``, ``env``, ``contact``,
        ``num``, ``power_setting``) or, via ``source_kwargs``, to the heat source.
    :returns: a ready model instance.
    """
    source_kwargs = kwargs.pop("source_kwargs", {})
    setup = FryingSetup(
        comp=Composition.from_row(FOODS.loc[food_id]),
        pan=PanMaterial.from_row(PANS.loc[pan_id]),
        source=HeatSource.from_row(SOURCES.loc[source_id], **source_kwargs),
        lethality=LETHALITY[lethality_key],
        **kwargs,
    )
    return FryingModel(setup)

## 8. Sprawdzenie założeń modelu

Zanim cokolwiek policzymy, warto sprawdzić liczbowo, czy przyjęte uproszczenia są ważne dla
konkretnego zestawu parametrów. Trzy testy:

1. **Udział brzegów** — stosunek mocy wchodzącej przez pobocznicę do mocy wchodzącej dnem.
   Model 1D jest sensowny, gdy jest to znacznie poniżej 5 %.
2. **Liczba Biota mięsa** $\mathrm{Bi} = h_c L / k$ — jeżeli $\mathrm{Bi} \gg 1$, model skupiony
   („kotlet ma jedną temperaturę”) jest niedopuszczalny i potrzebne jest rozwiązanie pola.
3. **Czas charakterystyczny dyfuzji** $t_c = L^2/\alpha$ — rząd wielkości czasu smażenia.

In [ ]:
def assumption_report(model: FryingModel) -> pd.DataFrame:
    """Dimensionless checks that justify (or refute) the one-dimensional treatment."""
    s = model.s
    L, R = s.geom.thickness_m, s.geom.radius_m
    k20 = s.comp.k(20.0)
    alpha = s.comp.alpha(20.0)

    ## Ratio of the power entering through the rim to the power entering through the base
    area_ratio = 2.0 * np.pi * R * L / s.geom.area_meat
    flux_ratio = area_ratio * s.env.h_top / s.contact.h_contact

    rows = [
        ("Edge-to-base power ratio", flux_ratio, "-", "< 0.05 justifies the 1D treatment"),
        ("Biot number of the meat", s.contact.h_contact * L / k20, "-",
         "> 10 means a lumped model is invalid"),
        ("Diffusion time L^2/alpha", L**2 / alpha, "s", "order of magnitude of the cooking time"),
        ("Node Fourier number at dt=1 s", alpha * 1.0 / model.dz**2, "-",
         "implicit solver, so no stability limit"),
        ("Pan Biot number (base)", s.env.h_pan_free * s.geom.pan_thickness_m / s.pan.k, "-",
         "<< 1 justifies lumping the pan across its thickness"),
    ]
    return pd.DataFrame(rows, columns=["quantity", "value", "unit", "interpretation"])


reference_model = build_model("pork_loin_lean", "cast_iron", "gas_burner_4980")
assumption_report(reference_model).round(4)

## 9. Zbieżność siatki

Rozwiązanie musi być niezależne od liczby węzłów. Sprawdzamy czas dojścia rdzenia do 63 °C,
końcową temperaturę powierzchni dolnej i ubytek wody. Przy $n = 41$ czas dojścia jest już
zbieżny z dokładnością do sekundy względem $n = 161$; dla przemiatania parametrów
wystarcza $n = 21$, co skraca obliczenia około pięciokrotnie.

In [ ]:
convergence = []
for n_nodes in (11, 21, 41, 81, 161):
    model = build_model("pork_loin_lean", "cast_iron", "gas_burner_4980",
                        num=NumericalSetup(n_nodes=n_nodes), power_setting=0.19)
    result = model.run(420.0, flips=(90.0, 180.0, 270.0, 360.0), T_pan0=210.0)
    convergence.append({
        "n_nodes": n_nodes,
        "t_core_63C_s": result.time_to_core(63.0),
        "T_bottom_end_C": result.T_bottom[-1],
        "water_loss_%": 100 * result.water_loss_fraction[-1],
    })
pd.DataFrame(convergence).set_index("n_nodes").round(2)

## 10. Parametry scenariusza

To jest jedyna komórka, którą trzeba edytować, żeby policzyć własny przypadek.

In [ ]:
# --- What is being fried -------------------------------------------------------------
FOOD_ID = "pork_loin_lean"        # see FOODS.index
PAN_ID = "cast_iron"              # see PANS.index
SOURCE_ID = "gas_burner_4980"     # see SOURCES.index
LETHALITY_KEY = "beef_6_5log"     # use "chicken_7log" for poultry

# --- Geometry ------------------------------------------------------------------------
GEOMETRY = Geometry(
    thickness_m=0.015,            # cutlet thickness
    radius_m=0.045,               # cutlet radius (equivalent disc)
    pan_radius_m=0.120,           # pan base radius
    pan_thickness_m=0.0045,       # pan base thickness
    pan_wall_height_m=0.045,
    coil_radius_m=0.090,          # induction coil / radiant element radius
)

# --- Operating conditions ------------------------------------------------------------
ENVIRONMENT = Environment(T_air_C=22.0, rh=0.45)
CONTACT = Contact(h_contact=900.0)
SOURCE_KWARGS = dict(distance_m=0.020, gap_m=0.0003)   # burner stand-off / hob gap

T_MEAT_INITIAL_C = 6.0            # straight from the fridge
T_PAN_TARGET_C = 210.0            # preheat target, measured on the empty pan
POWER_SETTING = 0.35              # knob position as a fraction of the nominal input rate
TOTAL_TIME_S = 480.0
FLIP_INTERVAL_S = 60.0

# --- Doneness targets ----------------------------------------------------------------
DONENESS = {"rare": 52.0, "medium-rare": 55.0, "medium": 60.0,
            "medium-well": 65.0, "well-done": 71.0}
CORE_TARGET_C = 63.0              # pork, medium

In [ ]:
model = build_model(FOOD_ID, PAN_ID, SOURCE_ID, LETHALITY_KEY, geom=GEOMETRY, env=ENVIRONMENT,
                    contact=CONTACT, source_kwargs=SOURCE_KWARGS, power_setting=POWER_SETTING)

flips = tuple(np.arange(FLIP_INTERVAL_S, TOTAL_TIME_S, FLIP_INTERVAL_S))
print(f"source            : {model.s.source.name}")
print(f"power setting     : {100 * POWER_SETTING:.1f} % of {model.s.source.power_W:.0f} W nominal")
print(f"delivered to pan  : {model.P_useful:.0f} W")
print(f"fraction under the cutlet: {model.f_inner:.3f}")
print(f"lateral conductance of the pan base: {model.G_lat:.2f} W/K")
print(f"flips at          : {', '.join(f'{t:.0f} s' for t in flips)}")
print(f"empty-pan steady state at this setting: "
      f"{model.steady_pan_temperature(POWER_SETTING):.0f} deg C")
print(f"setting that would hold the empty pan at {T_PAN_TARGET_C:.0f} deg C: "
      f"{100 * model.setting_for_pan_temperature(T_PAN_TARGET_C):.1f} %")

preheat = model.preheat(t_end=900.0)
result = model.run(TOTAL_TIME_S, flips=flips, T_meat0=T_MEAT_INITIAL_C, T_pan0=T_PAN_TARGET_C)

## 11. Wyniki

### 11.1. Nagrzewanie pustej patelni

Krzywa daje dwie liczby: czas potrzebny na dojście pustej patelni do temperatury docelowej
(od tego momentu startuje właściwa symulacja) oraz temperaturę równowagi przy zadanym
ustawieniu mocy. Ta druga jest zwykle znacznie wyższa od docelowej — to ilościowe uzasadnienie
tego, że pustej patelni nie zostawia się na ogniu: przy ustawieniu dobrym do smażenia
temperatura równowagi pustej patelni przekracza punkt dymienia typowych olejów (rzepakowy
rafinowany ok. 200–230 °C, masło klarowane ok. 250 °C).

In [ ]:
fig, ax = plt.subplots()
ax.plot(preheat.t / 60.0, preheat.T_pan_out, color=SERIES_COLORS[0], label="Pan, empty")
ax.axhline(T_PAN_TARGET_C, color="#666666", ls="--", lw=1.2)
ax.annotate(f"preheat target {T_PAN_TARGET_C:.0f} deg C",
            xy=(preheat.t[-1] / 60.0, T_PAN_TARGET_C), xytext=(-6, 6),
            textcoords="offset points", ha="right", color="#444444")

reached = np.flatnonzero(preheat.T_pan_out >= T_PAN_TARGET_C)
if reached.size:
    t_preheat = preheat.t[reached[0]]
    ax.plot([t_preheat / 60.0], [T_PAN_TARGET_C], "o", color=SERIES_COLORS[1], ms=7, zorder=5)
    ax.annotate(f"reached after {t_preheat:.0f} s", xy=(t_preheat / 60.0, T_PAN_TARGET_C),
                xytext=(8, -14), textcoords="offset points", color=SERIES_COLORS[1])
    print(f"preheat time to {T_PAN_TARGET_C:.0f} deg C: {t_preheat:.0f} s")
ax.set_xlabel("Time  [min]")
ax.set_ylabel("Pan temperature  [deg C]")
ax.set_title(f"Empty-pan warm-up: {model.s.pan.name} on {model.s.source.name}")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

### 11.2. Pole temperatury w kotlecie

Mapa pokazuje temperaturę w funkcji czasu i położenia w przekroju kotleta, $z = 0$ to
powierzchnia stykająca się z patelnią w chwili początkowej. Pionowe linie to obroty —
po każdym z nich gorąca powierzchnia trafia do góry i zaczyna stygnąć, a zimna na patelnię.
Izoterma 100 °C wyznacza front odparowania, czyli granicę skórki; izoterma docelowa pokazuje,
kiedy strefa wysmażenia dotrze do środka.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 4.2))
extent = [result.t[0] / 60.0, result.t[-1] / 60.0, 0.0, 1000 * GEOMETRY.thickness_m]
mesh = ax.imshow(result.T, origin="lower", aspect="auto", extent=extent,
                 cmap=SEQUENTIAL_CMAP, vmin=0.0, vmax=max(160.0, result.T.max()))
contours = ax.contour(result.t / 60.0, 1000 * model.z, result.T,
                      levels=[CORE_TARGET_C, 100.0], colors=["#ffffff", "#9ad5ff"],
                      linewidths=1.4, linestyles=["-", "--"])
ax.clabel(contours, fmt={CORE_TARGET_C: f"{CORE_TARGET_C:.0f} C", 100.0: "100 C"}, fontsize=8)
for flip_time in flips:
    ax.axvline(flip_time / 60.0, color="#ffffff", lw=0.8, alpha=0.55)
ax.set_xlabel("Time  [min]")
ax.set_ylabel("Height above the pan  [mm]")
ax.set_title("Temperature field in the cutlet (white verticals = flips)")
ax.grid(False)
bar = fig.colorbar(mesh, ax=ax, pad=0.02)
bar.set_label("Temperature  [deg C]")
fig.tight_layout()
plt.show()

### 11.3. Historie temperatur

Cztery wielkości na jednej osi, bo wszystkie są temperaturami w tych samych jednostkach:
powierzchnia dolna, środek, najzimniejszy punkt oraz obie temperatury patelni. Widać trzy
rzeczy, których model bez parowania by nie dał:

- powierzchnia stykająca się z patelnią zatrzymuje się w okolicy 100–110 °C, dopóki jest
  w niej woda wolna — to jest plateau odparowania i dlatego mokre mięso się nie rumieni;
- patelnia pod kotletem załamuje się o kilkadziesiąt stopni w kilkanaście sekund, po czym
  powoli wraca; różnica między patelnią pod kotletem a resztą dna to miara „sprężystości
  cieplnej” patelni;
- najzimniejszy punkt przez pierwszą minutę leży na *górnej* powierzchni, chłodzonej
  parowaniem, a nie w geometrycznym środku.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 4.4))
ax.plot(result.t / 60.0, result.T_bottom, color=SERIES_COLORS[1], label="Cutlet, pan-side face")
ax.plot(result.t / 60.0, result.T_centre, color=SERIES_COLORS[0], label="Cutlet, mid-plane")
ax.plot(result.t / 60.0, result.T_core, color=SERIES_COLORS[2], label="Cutlet, coldest point")
ax.plot(result.t / 60.0, result.T_pan_in, color=SERIES_COLORS[3], ls="--",
        label="Pan, under the cutlet")
ax.plot(result.t / 60.0, result.T_pan_out, color=SERIES_COLORS[4], ls="--",
        label="Pan, outside the cutlet")
ax.axhline(CORE_TARGET_C, color="#666666", lw=1.0, ls=":")
ax.annotate(f"core target {CORE_TARGET_C:.0f} deg C", xy=(0.02, CORE_TARGET_C + 3),
            xycoords=("axes fraction", "data"), color="#444444")
for flip_time in flips:
    ax.axvline(flip_time / 60.0, color="#cccccc", lw=0.8)
ax.set_xlabel("Time  [min]")
ax.set_ylabel("Temperature  [deg C]")
ax.set_title("Temperature histories")
ax.legend(loc="lower right", ncols=2)
fig.tight_layout()
plt.show()

### 11.4. Woda, skórka i bezpieczeństwo

Trzy różne wielkości, więc trzy osobne panele — nie dwie osie na jednym wykresie.
Wskaźnik brązowienia jest wielkością *względną*: całką arrheniusowską z energią aktywacji
120 kJ/mol, znormalizowaną tak, że minuta suchej powierzchni w 160 °C daje 1. Nadaje się do
porównywania wariantów, nie do przewidywania barwy w wartościach bezwzględnych.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6))

axes[0].plot(result.t / 60.0, 100 * result.water_loss_fraction, color=SERIES_COLORS[0],
             label="Evaporated water")
axes[0].set_ylabel("Mass loss  [% of initial mass]")
axes[0].set_title("Water loss by evaporation")

axes[1].plot(result.t / 60.0, result.Y[6 + result.n], color=SERIES_COLORS[1], label="Face A")
axes[1].plot(result.t / 60.0, result.Y[7 + result.n], color=SERIES_COLORS[3], label="Face B")
axes[1].set_ylabel("Browning index  [-]")
axes[1].set_title("Maillard browning of the two faces")

axes[2].plot(result.t / 60.0, np.clip(result.lethality, 0, 12), color=SERIES_COLORS[2],
             label="Accumulated lethality")
axes[2].axhline(model.s.lethality.n_log, color="#666666", ls="--", lw=1.2)
axes[2].annotate(f"{model.s.lethality.n_log:.1f} log10 required",
                 xy=(0.03, model.s.lethality.n_log + 0.3),
                 xycoords=("axes fraction", "data"), color="#444444")
axes[2].set_ylabel("log10 reductions at the coldest point")
axes[2].set_title("Microbial lethality")

for ax in axes:
    ax.set_xlabel("Time  [min]")
    ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 12. Porównania

### 12.1. Materiał patelni

Wszystkie patelnie startują z tej samej temperatury i dostają tę samą moc, więc porównujemy
wyłącznie ich odpowiedź na wrzucenie zimnego kotleta. Dwa panele rozdzielają dwa
zjawiska, które potocznie się myli:

- **lewy panel** — patelnie w typowych grubościach handlowych; tu wygrywa kombinacja
  pojemności cieplnej i przewodności, więc grube żeliwo wypada dobrze mimo przeciętnego $k$;
- **prawy panel** — te same materiały przy jednakowej grubości 3 mm; tu zostaje czyste
  $k$ i wygrywają miedź i aluminium.

Wniosek: „żeliwo lepiej trzyma temperaturę” jest w dużej mierze stwierdzeniem o grubości,
nie o materiale.

In [ ]:
def pan_response(pan_id: str, thickness_m: float | None = None, t_end: float = 180.0):
    """Simulate the pan-temperature dip caused by placing the cutlet, for one material."""
    thickness = (1e-3 * float(PANS.loc[pan_id, "thickness_mm_default"])
                 if thickness_m is None else thickness_m)
    geom = Geometry(**{**GEOMETRY.__dict__, "pan_thickness_m": thickness})
    pan_model = build_model(FOOD_ID, pan_id, SOURCE_ID, LETHALITY_KEY, geom=geom, env=ENVIRONMENT,
                            contact=CONTACT, source_kwargs=SOURCE_KWARGS,
                            num=NumericalSetup(n_nodes=21), power_setting=POWER_SETTING)
    return pan_model.run(t_end, T_meat0=T_MEAT_INITIAL_C, T_pan0=T_PAN_TARGET_C)


pan_ids = ["cast_iron", "carbon_steel", "stainless_304", "aluminium_3003", "copper"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=True)
records = []
for pan_id, color in zip(pan_ids, SERIES_COLORS):
    label = PANS.loc[pan_id, "name_pl"]
    for ax, thickness in zip(axes, (None, 0.003)):
        res = pan_response(pan_id, thickness)
        ax.plot(res.t, res.T_pan_in, color=color, label=label)
        if thickness is None:
            records.append({
                "pan": label,
                "thickness_mm": PANS.loc[pan_id, "thickness_mm_default"],
                "min_pan_T_C": res.T_pan_in.min(),
                "T_after_60s_C": res.T_pan_in[np.argmin(np.abs(res.t - 60.0))],
                "surface_T_after_180s_C": res.T_bottom[-1],
            })

axes[0].set_title("Commercial base thicknesses")
axes[1].set_title("All bases set to 3 mm")
for ax in axes:
    ax.set_xlabel("Time after placing the cutlet  [s]")
    ax.set_ylabel("Pan temperature under the cutlet  [deg C]")
axes[0].legend(loc="lower right")
fig.tight_layout()
plt.show()

pd.DataFrame(records).set_index("pan").round(1)

### 12.2. Źródło ciepła

Porównanie przy **tej samej mocy pobieranej przez urządzenie** i tej samej temperaturze
startowej patelni. Wtedy różnica sprawności przekłada się wprost na moc trafiającą do patelni,
a więc na tempo jej regeneracji i na czas smażenia. Palnik gazowy przy tym samym poborze mocy
dostarcza do patelni około 2,5 raza mniej niż indukcja — i tyle samo razy więcej energii
trafia do kuchni zamiast do jedzenia.

In [ ]:
source_ids = ["induction_3600", "electric_ceramic_2500", "electric_coil_2400", "gas_burner_4980"]
INPUT_POWER_W = 1200.0   # same electrical / gas input rate for every hob

fig, ax = plt.subplots(figsize=(9.0, 4.2))
source_records = []
for source_id, color in zip(source_ids, SERIES_COLORS):
    pan_for_source = PAN_ID
    if (SOURCES.loc[source_id, "requires_ferromagnetic"]
            and not PANS.loc[pan_for_source, "ferromagnetic"]):
        continue
    setting = min(1.0, INPUT_POWER_W / float(SOURCES.loc[source_id, "power_nominal_W"]))
    src_model = build_model(FOOD_ID, pan_for_source, source_id, LETHALITY_KEY, geom=GEOMETRY,
                            env=ENVIRONMENT, contact=CONTACT, source_kwargs=SOURCE_KWARGS,
                            num=NumericalSetup(n_nodes=21), power_setting=setting)
    res = src_model.run(TOTAL_TIME_S, flips=flips, T_meat0=T_MEAT_INITIAL_C,
                        T_pan0=T_PAN_TARGET_C)
    ax.plot(res.t / 60.0, res.T_pan_in, color=color, label=SOURCES.loc[source_id, "name_pl"])
    t_target = res.time_to_core(CORE_TARGET_C)
    source_records.append({
        "source": SOURCES.loc[source_id, "name_pl"],
        "setting_%": 100 * setting,
        "delivered_W": src_model.P_useful,
        "input_W": setting * src_model.s.source.power_W,
        "t_core_target_s": t_target,
        "energy_input_kJ": setting * src_model.s.source.power_W * t_target / 1000.0,
        "water_loss_%": 100 * res.water_loss_fraction[-1],
    })

ax.set_xlabel("Time  [min]")
ax.set_ylabel("Pan temperature under the cutlet  [deg C]")
ax.set_title(f"Pan recovery at {INPUT_POWER_W:.0f} W input, {PANS.loc[PAN_ID, 'name_pl']} "
             f"preheated to {T_PAN_TARGET_C:.0f} deg C")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

pd.DataFrame(source_records).set_index("source").round(1)

### 12.3. Mapa czasu smażenia

Czas dojścia rdzenia do temperatury docelowej w funkcji grubości kotleta i temperatury
wstępnego nagrzania patelni, przy stałym ustawieniu mocy. Temperatura patelni jest tu wyłącznie
warunkiem początkowym — tym, co realnie kontroluje kucharz, decydując, jak długo rozgrzewa
patelnię przed położeniem mięsa.

Grubość wchodzi w przybliżeniu kwadratowo (dyfuzja), temperatura wstępnego nagrzania — dużo
słabiej, bo zapas ciepła zgromadzony w dnie patelni wyczerpuje się w kilkanaście sekund,
a potem o wszystkim decyduje moc doprowadzana. To jest ilościowa wersja kuchennej zasady
„grubszego kotleta nie da się usmażyć szybciej mocniejszym ogniem — można go tylko spalić
z wierzchu”.

In [ ]:
thickness_mm_grid = np.array([8.0, 12.0, 16.0, 20.0, 25.0, 30.0])
pan_temp_grid = np.array([170.0, 190.0, 210.0, 230.0, 250.0])

time_map = np.full((thickness_mm_grid.size, pan_temp_grid.size), np.nan)
for i, thickness_mm in enumerate(thickness_mm_grid):
    geom_i = Geometry(**{**GEOMETRY.__dict__, "thickness_m": 1e-3 * thickness_mm})
    for j, pan_temp in enumerate(pan_temp_grid):
        model_ij = build_model(FOOD_ID, PAN_ID, SOURCE_ID, LETHALITY_KEY, geom=geom_i,
                               env=ENVIRONMENT, contact=CONTACT, source_kwargs=SOURCE_KWARGS,
                               num=NumericalSetup(n_nodes=21), power_setting=POWER_SETTING)
        horizon = 120.0 + 90.0 * thickness_mm
        res_ij = model_ij.run(horizon, flips=tuple(np.arange(60.0, horizon, 60.0)),
                              T_meat0=T_MEAT_INITIAL_C, T_pan0=pan_temp, dt_out=2.0)
        time_map[i, j] = res_ij.time_to_core(CORE_TARGET_C) / 60.0

fig, ax = plt.subplots(figsize=(7.5, 4.4))
mesh = ax.imshow(time_map, origin="lower", aspect="auto", cmap=SEQUENTIAL_CMAP,
                 extent=[pan_temp_grid[0] - 10, pan_temp_grid[-1] + 10,
                         thickness_mm_grid[0] - 2, thickness_mm_grid[-1] + 2.5])
for i, thickness_mm in enumerate(thickness_mm_grid):
    for j, pan_temp in enumerate(pan_temp_grid):
        ax.annotate(f"{time_map[i, j]:.1f}", (pan_temp, thickness_mm), ha="center", va="center",
                    color="#ffffff", fontsize=8)
ax.set_xticks(pan_temp_grid)
ax.set_yticks(thickness_mm_grid)
ax.set_xlabel("Preheated pan temperature  [deg C]")
ax.set_ylabel("Cutlet thickness  [mm]")
ax.set_title(f"Minutes to reach {CORE_TARGET_C:.0f} deg C at the coldest point "
             f"(flipped every 60 s)")
ax.grid(False)
bar = fig.colorbar(mesh, ax=ax, pad=0.02)
bar.set_label("Time  [min]")
fig.tight_layout()
plt.show()

### 12.4. Jak często obracać

Częste obracanie wyrównuje profil temperatury: żadna powierzchnia nie zostaje na patelni
dostatecznie długo, żeby się przegrzać, a obie połowy kotleta nagrzewają się na przemian.
Model pokazuje kompromis: krótki interwał skraca czas dojścia do temperatury docelowej
i obniża szczytową temperaturę powierzchni, ale zmniejsza też wskaźnik zbrązowienia, bo
powierzchnia nie ma czasu wyschnąć i przekroczyć 140 °C.

In [ ]:
flip_intervals = [15.0, 30.0, 60.0, 120.0, TOTAL_TIME_S / 2.0]
flip_records = []
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
for interval, color in zip(flip_intervals, SERIES_COLORS):
    model_f = build_model(FOOD_ID, PAN_ID, SOURCE_ID, LETHALITY_KEY, geom=GEOMETRY,
                          env=ENVIRONMENT, contact=CONTACT, source_kwargs=SOURCE_KWARGS,
                          num=NumericalSetup(n_nodes=21), power_setting=POWER_SETTING)
    res_f = model_f.run(TOTAL_TIME_S, flips=tuple(np.arange(interval, TOTAL_TIME_S, interval)),
                        T_meat0=T_MEAT_INITIAL_C, T_pan0=T_PAN_TARGET_C)
    label = f"every {interval:.0f} s" if interval < TOTAL_TIME_S / 2 else "one flip only"
    axes[0].plot(res_f.t / 60.0, res_f.T_core, color=color, label=label)
    axes[1].plot(res_f.t / 60.0, res_f.browning, color=color, label=label)
    flip_records.append({
        "flip interval": label,
        "t_core_target_s": res_f.time_to_core(CORE_TARGET_C),
        "peak_surface_T_C": max(res_f.T_bottom.max(), res_f.T_top.max()),
        "browning_index": res_f.browning[-1],
        "water_loss_%": 100 * res_f.water_loss_fraction[-1],
    })

axes[0].axhline(CORE_TARGET_C, color="#666666", ls=":", lw=1.0)
axes[0].set_ylabel("Coldest-point temperature  [deg C]")
axes[0].set_title("Core heating")
axes[1].set_ylabel("Browning index  [-]")
axes[1].set_title("Surface browning")
for ax in axes:
    ax.set_xlabel("Time  [min]")
    ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

pd.DataFrame(flip_records).set_index("flip interval").round(2)

## 13. Rekomendacja

Funkcja zbiera wszystko w jedną odpowiedź: kiedy wyjąć kotlet, czy jest bezpieczny, ile
straci wody i czy zdąży się zarumienić. Wskaźnik brązowienia czytamy pasmami: poniżej 0,3
powierzchnia jest blada, 0,3–3 to złoty kolor, 3–10 ciemny brąz, powyżej 10 spalenizna. Zwraca też czas, po którym spełniony jest *jednocześnie*
warunek temperatury docelowej i warunek letalności — dla drobiu te dwa warunki rozjeżdżają się
najbardziej.

In [ ]:
def browning_verdict(index: float) -> str:
    """Qualitative reading of the relative browning index."""
    bands = ((0.3, "pale, no crust"), (3.0, "golden"), (10.0, "dark brown"))
    return next((label for limit, label in bands if index < limit), "burnt")


def recommend(model: FryingModel, result: SimulationResult, core_target_C: float,
              doneness: dict[str, float]) -> pd.DataFrame:
    """Summarise one simulation as a cooking recommendation.

    :param model: the model that produced ``result``.
    :param result: the simulation output.
    :param core_target_C: target temperature of the coldest point.
    :param doneness: mapping of doneness names to coldest-point temperatures.
    :returns: a one-column table of findings.
    """
    t_target = result.time_to_core(core_target_C)
    n_required = model.s.lethality.n_log
    safe_idx = np.flatnonzero(result.lethality >= n_required)
    t_safe = float(result.t[safe_idx[0]]) if safe_idx.size else float("nan")
    t_ready = np.nanmax([t_target, t_safe])

    at = (np.argmin(np.abs(result.t - t_ready)) if np.isfinite(t_ready)
          else result.t.size - 1)
    rows = {
        "time to core target [min]": t_target / 60.0,
        f"time to {n_required:.1f} log10 lethality [min]": t_safe / 60.0,
        "recommended total time [min]": t_ready / 60.0,
        "recommended time per side [min]":
            t_ready / 60.0 / (1 + sum(1 for f in flips if f < t_ready)),
        "coldest point at that time [deg C]": result.T_core[at],
        "pan-side surface at that time [deg C]": result.T_bottom[at],
        "water lost [% of mass]": 100 * result.water_loss_fraction[at],
        "browning index [-]": result.browning[at],
        "browning verdict": browning_verdict(result.browning[at]),
        "doneness reached": next((name for name, T in sorted(doneness.items(),
                                                             key=lambda kv: -kv[1])
                                  if result.T_core[at] >= T), "raw"),
    }
    return pd.DataFrame({"value": rows.values()}, index=list(rows)).round(2)


recommend(model, result, CORE_TARGET_C, DONENESS)

## 14. Ograniczenia i kierunki rozwoju

**Wielkości skalibrowane, nie zmierzone.** Trzy parametry modelu są dobrane tak, żeby wyniki
mieściły się w obserwowanych zakresach, i nie pochodzą z pomiaru dla tego konkretnego układu:
$\dot m_{\max}$ (transport wody przez skórkę, $10^{-2}\ \mathrm{kg/(m^2 s)}$, kalibrowane na
ubytek masy 15–30 %), $\eta_{\text{naczynia}} = 0{,}90$ w rozkładzie sprawności oraz
normalizacja wskaźnika brązowienia. Wyniki bezwzględne zależą od nich liniowo; porównania
wariantów — dużo słabiej.

**$h_c$ jest najczulszym parametrem wejściowym.** Zakres literaturowy 200–1200 W/(m²·K) to
sześciokrotna zmienność, zależna od docisku, tłuszczu i równości powierzchni mięsa. Warto
przed użyciem modelu do wartości bezwzględnych przemiatać ten parametr.

**Czego brakuje, gdyby model miał być dokładniejszy:**

1. Kurczenie się i wyciek soku (denaturacja kolagenu powyżej ~60 °C) — obecnie cały ubytek
   masy to parowanie, więc model zaniża go o kilka do kilkunastu punktów procentowych.
2. Ruchomy front skórki z osobnymi właściwościami (model Sternera i in. rozwiązuje transport
   pary w skórce i wody w rdzeniu jako dwa obszary z ruchomą granicą).
3. Model 2D albo osiowosymetryczny, żeby dostać szybsze gotowanie brzegów.
4. Warstwa oleju jako osobne medium, z własną temperaturą i punktem dymienia.
5. Walidacja doświadczalna: termopara w środku kotleta i termometr na podczerwień na
   powierzchni patelni dałyby dane do dopasowania $h_c$ i $\dot m_{\max}$.

## Źródła danych

- ASHRAE Handbook — Refrigeration, rozdz. „Thermal Properties of Foods”: tabele 1–3
  (współczynniki Choi–Okos dla składników, skład podstawowy produktów).
- Choi Y., Okos M. R. (1986), *Effects of Temperature and Composition on the Thermal
  Properties of Foods*.
- USDA FoodData Central — skład schabu i piersi kurczaka.
- USDA FSIS Appendix A — tabele czas–temperatura dla redukcji *Salmonelli*.
- Dagerskog M. (1985), *Heat transfer during contact cooking of minced meat patties* —
  kontaktowy współczynnik przejmowania ciepła 200–1200 W/(m²·K).
- Sterner D., Sundén B., Skjöldebrand C. (2002), *Predictive modelling of heat and mass
  transfer in hamburger patties* — struktura modelu rdzeń/skórka z ruchomą granicą.
- Frontier Energy / City of Oakland (2019), *Residential Cooktop Performance and Energy
  Comparison Study*, tab. 4 — zmierzone sprawności palników i płyt.
- EngineeringToolBox, CenturyLife — właściwości cieplne i emisyjności materiałów naczyń.